# MiniHEP 01: learn from masked four-vectors

This is a small **self-supervised pretraining** exercise on the workshop's reduced ATLAS Open Data `dataWW_d1.root`. Each event becomes up to five tokens: two charged leptons, two leading jets, and a missing-transverse-momentum proxy. We hide whole token four-vectors and train a tiny transformer to reconstruct them from the other objects.

**Provenance:** dataWW_d1.root is the course repository simulation derived from ATLAS Open Data. Labels, identifiers, and MC weights are never model inputs.

## Setup

From the course repository root, install once with `python -m pip install -r minihep_tutorial/requirements.txt`. This notebook requires the repository `dataWW_d1.root` file.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch

TUTORIAL_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "minihep" / "model.py").exists())
sys.path.insert(0, str(TUTORIAL_ROOT))

from minihep.data import TYPE_NAMES, VectorScaler, load_workshop, split_events
from minihep.model import MiniHEP
from minihep.train import heldout_loss, pretrain

torch.set_num_threads(1)  # the workshop's CPU Binder has one core
torch.manual_seed(42)
MAX_EVENTS = 6000
candidates = [p / "dataWW_d1.root" for p in (Path.cwd(), *Path.cwd().parents)]
root_path = next((p for p in candidates if p.is_file()), None)
if root_path is None:
    raise FileNotFoundError("dataWW_d1.root is missing. Run from a course-repository checkout.")

events = load_workshop(root_path, max_events=MAX_EVENTS, seed=42)
print(f"SOURCE: {events.source}")
print(f"Selected events: {len(events.vectors):,}; valid tokens: {events.valid.sum():,}")
print("First event token types:", [TYPE_NAMES[t] for t in events.types[0]])
print("First event four-vectors (px, py, pz, E) in GeV:\n", np.round(events.vectors[0], 2))

## Four-vector conventions

The stored lepton and jet \(p_T\) values are in **GeV**, but their stored energies are in **MeV**. The loader converts energies to GeV and computes \((p_x,p_y,p_z,E)\). Jet slots with `jet_n` below their slot number are padding, even though the file puts `-7` in their numeric columns. The missing-momentum token is \((E_T^{miss}\cos\phi, E_T^{miss}\sin\phi, 0, E_T^{miss})\), a convenient transverse **proxy**, not a measured neutrino-system four-vector.

Electron/muon flavor and charge, jet, and MET are discrete token types with learned embeddings. The four continuous vector components are transformed with `asinh(component / 50 GeV)` and standardized using **training events only**.

In [ ]:
train_ids, val_ids, test_ids = split_events(len(events.vectors), seed=42)
assert len(np.unique(events.event_ids, axis=0)) == len(events.event_ids)
scaler = VectorScaler().fit(events.vectors, events.valid, train_ids)
x = scaler.transform(events.vectors, events.valid)
print(f"train / validation / test: {len(train_ids):,} / {len(val_ids):,} / {len(test_ids):,}")
print("Training-only feature mean:", np.round(scaler.mean, 3))
print("Training-only feature std: ", np.round(scaler.std, 3))

## Masked-token pretraining

The model has width 32, two attention layers, four heads, and dropout 0.1. Each training event hides about 25% of its **valid** tokens, with at least one hidden token. All four vector components of those tokens are removed from the input and scored in the loss. The attention padding mask excludes absent jets. A fixed masking sequence makes validation losses comparable across epochs.

In [ ]:
model = MiniHEP(width=32, heads=4, layers=2, dropout=0.1)
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters()):,}")
history = pretrain(model, x, events.types, events.valid,
                   train_ids, val_ids, epochs=4, batch_size=64,
                   mask_probability=0.25, seed=42)

## First diagnostic: held-out masked reconstruction loss

The curve is mean squared error in standardized four-vector components. It checks the training loop and overfitting; it does not by itself establish useful physics performance.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
epochs = np.arange(1, len(history.train) + 1)
ax.plot(epochs, history.train, "o-", label="train")
ax.plot(epochs, history.validation, "o-", label="validation")
ax.set(xlabel="epoch", ylabel="masked four-vector MSE", title=f"MiniHEP — {events.source}")
ax.set_xticks(epochs)
ax.grid(alpha=0.25)
ax.legend()
plt.show()

print(f"Final held-out test MSE: {heldout_loss(model, x, events.types, events.valid, test_ids):.4f}")

## Event representation and exercises

With every token visible, mean-pool the encoder output for an event representation. The optional next notebook can freeze this encoder and train a small signal/background probe using labels **only after pretraining**, with the same event split. Try changing the masking fraction or dropout and compare validation loss. For a physics diagnostic, compare the lepton-pair invariant-mass distribution and masked reconstruction quality across kinematic ranges.

In [ ]:
model.eval()
with torch.no_grad():
    embeddings = model.encode_events(
        torch.from_numpy(x[test_ids[:8]]),
        torch.from_numpy(events.types[test_ids[:8]]),
        torch.from_numpy(events.valid[test_ids[:8]]),
    )
print("Eight held-out event embeddings:", tuple(embeddings.shape))